# Plot Protein LM Training Metrics

Read `training_metrics.jsonl` and optional `resume_training_metrics.jsonl` from the protein checkpoint directory, then plot train and validation loss against optimizer steps.

In [ ]:
import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt

def find_repo_dir_for_import(start: Path) -> Path:
    candidates = [start.resolve(), *start.resolve().parents]
    repo_dir_env = os.environ.get("MDNAC_REPO_DIR")
    if repo_dir_env:
        candidates.append(Path(repo_dir_env).expanduser())
    candidates.extend([
        Path("/content/Microbial DNA Compiler"),
        Path("/content/Microbial-DNA-Compiler"),
        Path("/content/drive/MyDrive/Microbial DNA Compiler"),
        Path("/content/drive/MyDrive/Microbial-DNA-Compiler"),
    ])
    for candidate in candidates:
        resolved = candidate.expanduser().resolve()
        if (resolved / "pyproject.toml").exists() and (resolved / "libs").is_dir():
            return resolved
    raise RuntimeError(
        "Could not locate repo. Run inside the repo, set MDNAC_REPO_DIR, "
        "or in Colab clone/mount the repo under /content or Google Drive."
    )


PROJECT_ROOT = find_repo_dir_for_import(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from libs.notebook_runtime import bootstrap_notebook
from libs.core.pretrain.training_config import load_protein_training_config

RUNTIME = bootstrap_notebook(PROJECT_ROOT)
PROJECT_ROOT = Path(RUNTIME["repo_dir"])
CONFIG_PATH = Path(os.environ.get("MDNAC_TRAIN_CONFIG", PROJECT_ROOT / "train.16gb.yaml")).expanduser()
if not CONFIG_PATH.is_absolute():
    CONFIG_PATH = (PROJECT_ROOT / CONFIG_PATH).resolve()
TRAINING_CONFIG = load_protein_training_config(PROJECT_ROOT, config_path=CONFIG_PATH)
CHECKPOINT_DIR = TRAINING_CONFIG["paths"]["checkpoint_dir"]
METRIC_FILES = [
    CHECKPOINT_DIR / "metrics_history.jsonl",
    CHECKPOINT_DIR / "training_metrics.jsonl",
    CHECKPOINT_DIR / "resume_training_metrics.jsonl",
]
PLOT_PATH = CHECKPOINT_DIR / "loss_curve.png"

{"runtime": RUNTIME, "config_path": str(CONFIG_PATH), "checkpoint_dir": str(CHECKPOINT_DIR)}


In [ ]:
rows = []
for metric_file in METRIC_FILES:
    if not metric_file.exists():
        continue
    for line in metric_file.read_text(encoding="utf-8").splitlines():
        if line.strip():
            row = json.loads(line)
            row["metric_file"] = str(metric_file)
            rows.append(row)

if not rows:
    raise FileNotFoundError(
        f"No metric JSONL files found under {CHECKPOINT_DIR}. Run notebook 03 or 04 first."
    )

rows = sorted(rows, key=lambda item: int(item.get("global_step", 0)))
rows[:3], rows[-3:]

In [ ]:
steps = [int(row["global_step"]) for row in rows]
train_losses = [float(row["train_loss"]) for row in rows]
val_losses = [float(row["val_loss"]) for row in rows]
tokens_seen = [int(row.get("tokens_seen", 0)) for row in rows]

fig, ax1 = plt.subplots(figsize=(8, 4.5))
ax1.plot(steps, train_losses, label="Train loss", linewidth=2)
ax1.plot(steps, val_losses, label="Validation loss", linewidth=2)
ax1.set_title("Protein LM Training Metrics")
ax1.set_xlabel("Optimizer step")
ax1.set_ylabel("Loss")
ax1.grid(True, alpha=0.25)
ax1.legend()

ax2 = ax1.twiny()
ax2.plot(tokens_seen, train_losses, alpha=0)
ax2.set_xlabel("Tokens seen")

fig.tight_layout()
PLOT_PATH.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(PLOT_PATH, dpi=150)
PLOT_PATH